# 17 模型交付安全与 IP 保护

演示 **哈希完整性、HMAC 签名、简易权重水印、设备绑定 license、安全加载流水线**（教学级，非生产密码学库替代）。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass, field
from typing import Optional
import time
import hashlib
import hmac
import json
import math

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch {torch.__version__}")

import os
from pathlib import Path

## 17.1 完整性：SHA256 + HMAC 签名

In [ ]:
def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sign(data: bytes, key: bytes) -> str:
    return hmac.new(key, data, hashlib.sha256).hexdigest()


def verify(data: bytes, key: bytes, signature: str) -> bool:
    return hmac.compare_digest(sign(data, key), signature)


# 模拟模型文件
weights = torch.randn(64, 128).numpy().astype(np.float16).tobytes()
vendor_key = b"vendor-secret-demo-key"
digest = sha256_bytes(weights)
sig = sign(weights, vendor_key)
print("sha256:", digest[:16] + "...")
print("verify ok:", verify(weights, vendor_key, sig))
tampered = weights[:-1] + bytes([(weights[-1] + 1) % 256])
print("tamper detect:", verify(tampered, vendor_key, sig))

## 17.2 权重水印（可溯源）

In [ ]:
def embed_watermark(w: torch.Tensor, payload_bits: str, scale: float = 1e-4) -> torch.Tensor:
    """在最小幅度系数上叠加 ±scale 作为比特水印。"""
    flat = w.reshape(-1).clone()
    n = min(len(payload_bits), flat.numel())
    idx = torch.arange(n)
    bits = torch.tensor([1.0 if b == "1" else -1.0 for b in payload_bits[:n]])
    flat[idx] = flat[idx] + bits * scale
    return flat.view_as(w)


def extract_watermark(w: torch.Tensor, base: torch.Tensor, n_bits: int, scale: float = 1e-4) -> str:
    delta = (w - base).reshape(-1)[:n_bits]
    return "".join("1" if d > 0 else "0" for d in delta)


payload = "1011001110001101"
base = torch.randn(32, 32) * 0.02
marked = embed_watermark(base, payload)
recovered = extract_watermark(marked, base, len(payload))
print("payload   :", payload)
print("recovered :", recovered)
print("match:", payload == recovered)

## 17.3 设备绑定 License

In [ ]:
def device_fingerprint(device_id: str, app_id: str) -> str:
    return hashlib.sha256(f"{device_id}|{app_id}".encode()).hexdigest()[:32]


def issue_license(device_id: str, app_id: str, model_id: str, key: bytes, ttl_days: int = 30) -> dict:
    fp = device_fingerprint(device_id, app_id)
    body = f"{fp}|{model_id}|{ttl_days}".encode()
    return {"fingerprint": fp, "model_id": model_id, "ttl_days": ttl_days, "sig": sign(body, key)}


def check_license(lic: dict, device_id: str, app_id: str, key: bytes) -> bool:
    fp = device_fingerprint(device_id, app_id)
    if fp != lic["fingerprint"]:
        return False
    body = f"{fp}|{lic['model_id']}|{lic['ttl_days']}".encode()
    return verify(body, key, lic["sig"])


lic = issue_license("pixel-demo-001", "com.demo.assistant", "qwen3-4b-w4", vendor_key)
print("same device:", check_license(lic, "pixel-demo-001", "com.demo.assistant", vendor_key))
print("other device:", check_license(lic, "pixel-hacked", "com.demo.assistant", vendor_key))

## 17.4 安全加载流水线（伪代码可运行）

In [ ]:
class SecureModelLoader:
    def __init__(self, vendor_key: bytes):
        self.vendor_key = vendor_key

    def load(self, blob: bytes, signature: str, lic: dict, device_id: str, app_id: str) -> dict:
        if not check_license(lic, device_id, app_id, self.vendor_key):
            return {"ok": False, "error": "license_denied"}
        if not verify(blob, self.vendor_key, signature):
            return {"ok": False, "error": "integrity_failed"}
        # 生产中：在 TEE 内解密；这里仅演示通过
        return {"ok": True, "bytes": len(blob), "sha16": sha256_bytes(blob)[:16]}


loader = SecureModelLoader(vendor_key)
print(loader.load(weights, sig, lic, "pixel-demo-001", "com.demo.assistant"))
print(loader.load(tampered, sig, lic, "pixel-demo-001", "com.demo.assistant"))

## 小结

1. **完整性**先于加密：先发现篡改。
2. 水印用于溯源，不能单独当 DRM。
3. License 绑定设备指纹，配合 OTA 吊销。
4. 生产请用平台 Keystore / TEE / 厂商 Secure Model Container，不要自己发明加密协议。